# Reconstrução do Universo Investível — Aplicação das Exclusões de Auditoria

Este notebook consome a saída da construção mecânica do universo (`03_build_investable_universe_of_stocks.ipynb`)
e da auditoria de `codigo_cvm` (`04_audit.ipynb`) para produzir a versão **final** do universo investível,
já com as exclusões justificadas e documentadas.

Nenhuma exclusão é aplicada silenciosamente: cada ticker removido tem uma linha correspondente na tabela
de exclusões (`08_audit_exclusions/excluded_tickers.csv`), com a categoria do motivo e a evidência que
sustentou a decisão. O objetivo é que qualquer pessoa (inclusive eu, revisando isso daqui a um ano) consiga
entender e reproduzir cada corte sem precisar reconstruir o raciocínio a partir do código.

**Camadas de exclusão aplicadas, em ordem:**
1. Ativos sem nenhum demonstrativo DFP/ITR no período em que foram negociados (`SEM_DFP_OU_ITR`).
2. Ativos com `codigo_cvm` no diretório `CIA_ESTRANGEIRA` da CVM (prefixo `80`) — fora do escopo do estudo.
3. Ativos remanescentes com nome divergente ou código inexistente no cadastro — revisão manual pontual.

In [9]:
import pandas as pd
import json
from pathlib import Path

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)

caminho_atual = Path.cwd()
pasta_anterior = caminho_atual.parent
pasta_interim = pasta_anterior / "data/interim"

## Passo 1 — Carregar o mapeamento bruto e o resultado da auditoria

`05_ticker_and_code_cvm_mapping.csv` vem de `03` (todo ticker considerado ação negociável, sem
filtro de identidade da companhia). `07_audit/complete_code_cvm.csv` vem de `04` e traz o `status_final`
de cada ticker — incluindo o status `SEM_DFP_OU_ITR`, atribuído manualmente ao final de `04` para os
tickers cujo `codigo_cvm` está correto mas não possui nenhum demonstrativo disponível no período em
que o ticker foi negociado.

In [10]:
mapping = pd.read_csv(pasta_interim / "05_investable_universe_of_stocks/ticker_and_code_cvm_mapping.csv", sep=";", dtype=str)
mapping["TICKER"] = mapping["TICKER"].str.strip().str.upper()

auditoria = pd.read_csv(pasta_interim / "07_audit/complete_code_cvm.csv", sep=";", dtype=str)
auditoria["TICKER"] = auditoria["TICKER"].str.strip().str.upper()

base = mapping.merge(
    auditoria[["TICKER", "status_final", "DENOM_SOCIAL", "score_nome_cadastro"]],
    on="TICKER", how="left"
)

print(f"Total de tickers no universo bruto (pós-03): {len(base)}")
print("\nDistribuição de status_final:")
print(base["status_final"].value_counts(dropna=False))

Total de tickers no universo bruto (pós-03): 774

Distribuição de status_final:
status_final
ALTA_VIA_FCA                  547
ALTA_VIA_CADASTRO             130
REVISAR_NOME_DIVERGENTE        84
SEM_DFP_OU_ITR                 12
REVISAR_CODIGO_INEXISTENTE      1
Name: count, dtype: int64


## Passo 2 — Exclusão 1: ativos sem DFP/ITR no período observado

Estes são tickers cujo `codigo_cvm` foi confirmado como correto (Camada 1 ou 2 da auditoria em `04`),
mas para os quais não existe nenhum demonstrativo financeiro (DFP ou ITR) nos anos em que o ticker foi
efetivamente negociado. Isso não é um erro de mapeamento — é uma limitação de cobertura de dados da CVM
para o período 2010–2025. Excluímos por esse motivo, e não por qualquer critério ligado a desempenho do
ativo, para não introduzir viés de sobrevivência: a exclusão está condicionada exclusivamente à
disponibilidade de dado, não ao resultado da empresa no mercado.

In [11]:
excluidos_sem_dfp = base[base["status_final"] == "SEM_DFP_OU_ITR"].copy()
excluidos_sem_dfp["motivo_categoria"] = "SEM_DFP_OU_ITR"
excluidos_sem_dfp["motivo_detalhado"] = (
    "codigo_cvm confirmado correto, porém sem nenhum demonstrativo DFP/ITR "
    "disponível no período em que o ticker foi negociado."
)
excluidos_sem_dfp["fonte_evidencia"] = "04_audit.ipynb — cruzamento com 02_dfp_concatenated/02_itr_concatenated"

print(f"Excluídos por ausência de DFP/ITR: {len(excluidos_sem_dfp)}")
display(excluidos_sem_dfp[["TICKER", "codigo_cvm", "primeiro_ano_observado", "ultimo_ano_observado"]]
        .sort_values("TICKER"))

base_pos_dfp = base[base["status_final"] != "SEM_DFP_OU_ITR"].copy()
print(f"\nRestam no universo: {len(base_pos_dfp)}")

Excluídos por ausência de DFP/ITR: 12


,TICKER,codigo_cvm,primeiro_ano_observado,ultimo_ano_observado
6,ABYA3,20206,2010,2010
7,ACGU3,20940,2010,2010
17,AGEI3,21911,2010,2010
55,AVIL3,108,2010,2011
362,GVTT3,20117,2010,2010
426,KSSA3,20249,2010,2010
456,MARI3,20761,2010,2010
463,MEDI3,20273,2010,2010
667,SZPQ4,19267,2010,2010
723,TVIT3,21768,2010,2010



Restam no universo: 762


## Passo 3 — Casos remanescentes de revisão manual (`REVISAR_NOME_DIVERGENTE` / `REVISAR_CODIGO_INEXISTENTE`)

Estes tickers não foram automaticamente validados na Camada 1 (FCA) nem na Camada 2 (cadastro geral) do
notebook `04`: o nome do COTAHIST não bateu com a razão social cadastrada acima do limiar de similaridade
definido, ou o `codigo_cvm` sequer existe no cadastro. Isso pode significar erro de digitação no
`codigo_cvm`, mudança de razão social não capturada pelo cadastro reduzido (`keep="last"`), ou — como no
caso de `PPLA11` — um ativo cujo registro segue outro regime (estrangeiro) e por isso não aparece no
cadastro nacional. A lista abaixo precisa de checagem individual antes de decidir manter ou excluir.

In [12]:
para_revisao = base_pos_dfp[
    base_pos_dfp["status_final"].isin(["REVISAR_NOME_DIVERGENTE", "REVISAR_CODIGO_INEXISTENTE"])
].sort_values("score_nome_cadastro")

print(f"--- {len(para_revisao)} tickers pendentes de revisão manual ---")
display(para_revisao[[
    "TICKER", "codigo_cvm", "status_final", "DENOM_SOCIAL", "score_nome_cadastro",
    "primeiro_ano_observado", "ultimo_ano_observado"
]])

--- 85 tickers pendentes de revisão manual ---


,TICKER,codigo_cvm,status_final,DENOM_SOCIAL,score_nome_cadastro,primeiro_ano_observado,ultimo_ano_observado
557,PPLA11,80152,REVISAR_CODIGO_INEXISTENTE,NaN,0.0,2017,2022
746,VIVO3,17671,REVISAR_NOME_DIVERGENTE,TELEFÔNICA BRASIL S.A.,25.0,2010,2011
747,VIVO4,17671,REVISAR_NOME_DIVERGENTE,TELEFÔNICA BRASIL S.A.,25.0,2010,2011
273,ECOD3,20354,REVISAR_NOME_DIVERGENTE,TERRA SANTA AGRO S.A.,28.57142857142857,2010,2011
694,TIBR5,11398,REVISAR_NOME_DIVERGENTE,TRONOX PIGMENTOS DO BRASIL S.A.,30.000000000000004,2010,2015
695,TIBR6,11398,REVISAR_NOME_DIVERGENTE,TRONOX PIGMENTOS DO BRASIL S.A.,30.000000000000004,2010,2014
3,ABNB3,20028,REVISAR_NOME_DIVERGENTE,VALID SOLUÇÕES S.A.,33.333333333333336,2010,2010
451,LVTC3,25895,REVISAR_NOME_DIVERGENTE,LIVETECH DA BAHIA INDÚSTRIA E COMÉRCIO S.A.,33.333333333333336,2021,2025
517,OHLB3,19771,REVISAR_NOME_DIVERGENTE,ARTERIS S.A.,33.333333333333336,2010,2012
760,WDCN3,25895,REVISAR_NOME_DIVERGENTE,LIVETECH DA BAHIA INDÚSTRIA E COMÉRCIO S.A.,33.333333333333336,2025,2025


## Passo 4 — Exclusão manual pontual

Após a checagem individual da lista acima (comparando com o histórico de razão social da CVM, notícias
de fusões/incorporações, ou o próprio ISIN), os tickers abaixo foram avaliados e considerados incorretos
ou inadequados ao universo investível. A lista começa vazia — cada item incluído aqui deve vir acompanhado,
fora deste notebook (na tabela de exclusões final), do motivo específico da decisão.

In [ ]:
# Preencher conforme a revisão manual da lista do Passo 4 avança.
# Formato: "TICKER" -> motivo textual da exclusão
EXCLUSAO_MANUAL = {
    "PPLA11": "A PPLA Participations Ltd. é uma empresa estrangeira. Não é ação brasileira",
}

tickers_excluidos = {
    "PPLA11": "Não corresponde a uma ação brasileira, pois trata-se de um BDR de uma holding sediada nas Bermudas",

}

tickers_mantidos = {
    "VIVO3": "Telefônica Brasil S.A. (antiga Telesp / Vivo)",
    "VIVO4": "Telefônica Brasil S.A. (antiga Telesp / Vivo)",
    "ECOD3": "Brasil Ecodiesel Ind. e Com. de Biocombustíveis e Óleos Vegetais S.A. (posteriormente alterada para Vanguarda Agro / V-Agro)",
    "TIBR5": "Cristal Pigmentos do Brasil S.A. (anteriormente denominada Tibras - Titânio do Brasil S.A. e Millennium Inorganic Chemicals do Brasil S.A., atualmente conhecida como Tronox Pigmentos do Brasil S.A.)",
    "TIBR6": "Cristal Pigmentos do Brasil S.A. (anteriormente denominada Tibras - Titânio do Brasil S.A. e Millennium Inorganic Chemicals do Brasil S.A., atualmente conhecida como Tronox Pigmentos do Brasil S.A.)",
    "ABNB3": "Valid Soluções S.A. (Anteriormente chamada ABnote)",
    "LVTC3": "Livetech da Bahia Indústria e Comércio S.A. (Conhecida comercialmente como WDC Networks)",
    "OHLB3": "Arteris S.A. (Anteriormente denominada Obrascon Huarte Lain Brasil - OHL Brasil)",
    "WDCN3": "Livetech da Bahia Indústria e Comércio S.A. (WDC Networks)",
    "BTTL3": "Embpar Participações S.A. (Histórica Battistella Administração e Participações S.A.)",
    "BTTL4": "Embpar Participações S.A. (Histórica Battistella Administração e Participações S.A.)",
    "GLOB3": "Globex Utilidades S.A. (antiga dona do Ponto Frio)",
    "SWET3": "Sweet Cosmetics S.A. (antiga All Ore Mineração S.A.)",
    "ADMF3": "B100 S.A. (anteriormente denominada Ciabrasf Cia Brasileira de Serviços Financeiros S.A.)",
    "DMMO3": "Dommo Energia S.A. (nova denominação da histórica OGX Petróleo e Gás S.A.)",
    "HRTP3": "HRT Participações em Petróleo S.A. (que depois mudou de nome para PetroRio S.A. e, posteriormente, para PRIO)",
    "KROT11": "Kroton Educacional S.A. (atualmente denominada Cogna Educação S.A.)",
    "KROT3": "Kroton Educacional S.A. (atualmente denominada Cogna Educação S.A.)",
    "RNAR3": "Pomifrutas S.A. (Historicamente conhecida pelo nome de prego Renar Maçãs S.A.)",
    "BRTO3": "OI S.A.",
    "BRTO4": "OI S.A.",
    "LFFE3": "Originalmente La Fonte Telecom S.A. e, em dezembro de 2016, alterou a sua denominação social para JPSP Investimentos e Participações S.A",
    "LFFE4": "Originalmente La Fonte Telecom S.A. e, em dezembro de 2016, alterou a sua denominação social para JPSP Investimentos e Participações S.A",
    "STLB3": "Advanced Digital Health Medicina Preventiva S.A. (Originalmente listada como Steel do Brasil Participações S.A., que posteriormente alterou o objeto e denominação social para All Ore Mineração, depois Sweet Cosmetics e, por fim, ADH",
    "PCAR5": "Companhia Brasileira de Distribuição (Grupo Pão de Açúcar - GPA)",
    "BNCA3": "Banco Nossa Caixa S.A.. O Banco do Brasil S.A. foi o banco que comprou e incorporou a instituição.",
    "OGSA3": "Dommo Energia S.A. (Denominada na época do ticker como OGX Petróleo e Gás S.A.)",
    "TBLE3": "Engie Brasil Energia S.A. (Conhecida historicamente no pregão como Tractebel Energia S.A.)",
    "ALLL11": "Rumo S.A. (Denominada no período como ALL - América Latina Logística S.A.)",
    "ALLL3": "Rumo S.A. (Denominada no período como ALL - América Latina Logística S.A.)",
    "ALLL4": "Rumo S.A. (Denominada no período como ALL - América Latina Logística S.A.)",
    "INET3": "O ticker INET3 pertenceu originalmente à Inepar Telecomunicações S.A.. Contudo, por meio de profundos processos de reestruturação acionária e IPOs reversos, essa mesma casca jurídica passou a ser a Atom Empreendimentos e, recentemente, foi rebatizada para FICTOR ALIMENTOS S.A. - EM RECUPERAÇÃO JUDICIAL.",
    "JBDU11": "Blue Tech Solutions E.Q.I. S.A. (Historicamente conhecida de ponta a ponta como Indústrias J.B. Duarte S.A.)",
    "JBDU4": "Blue Tech Solutions E.Q.I. S.A. (Historicamente conhecida de ponta a ponta como Indústrias J.B. Duarte S.A.)",
    "JPSA4": "Jereissati Participações S.A. (atualmente unificada à Iguatemi S.A.)",
    "MLFT3": "Jereissati Participações S.A. (antiga denominação de pregão Iguatemi/La Fonte)",
    "MLFT4": "Jereissati Participações S.A. (antiga denominação de pregão Iguatemi/La Fonte)",
    "BPNM4": "Banco PAN S.A. (Historicamente denominado no período como Banco PanAmericano S.A.)",
    "BRIN3": " companhia chamava-se originalmente BR Insurance Corretora de Seguros S.A., mas alterou oficialmente a sua denominação social em Assembleia Geral para ALPER CONSULTORIA E CORRETORA DE SEGUROS S.A. (com nome de pregão reduzido para Alper S.A.)",
    "CLSC6": "Centrais Elétricas de Santa Catarina S.A. (CELESC)",
    "MPXE3": "Eneva S.A. (Anteriormente chamada MPX Energia S.A., do antigo grupo EBX)",
    "PTPA4": "Évora S.A. (Historicamente denominada Petropar S.A., alterou o nome em pregão e razão social em 2014).",
    "TRNA11": "Transmissora Aliança de Energia Elétrica S.A. - TAESA (Originalmente listada como Terna Participações S.A.)",
    "SGAS3": "WLM Indústria e Comércio S.A. (Conhecida anteriormente como Saga Administração e Participações S.A.)",
    "SGAS4": "WLM Indústria e Comércio S.A. (Conhecida anteriormente como Saga Administração e Participações S.A.)",
    "ABRE11": "Arco Platform / Estrutura histórica da Abril Educação S.A. (Companhia que posteriormente foi rebatizada para Somo Educação S.A. antes de seu fechamento de capital)",
    "ABRE3": "Arco Platform / Estrutura histórica da Abril Educação S.A. (Companhia que posteriormente foi rebatizada para Somo Educação S.A. antes de seu fechamento de capital)",
    "CCIM3": "Camargo Corrêa Desenvolvimento Imobiliário S.A. - CCDI",
    "VTLM3": "Vitalyze.me Saúde e Tecnologia S.A. (Empresa que herdou a casca jurídica da Steel do Brasil / All Ore / Sweet Cosmetics e adotou temporariamente este ticker em 2016 antes de passar por novas reestruturações)",
    "AORE3": "All Ore Mineração S.A. (Fase corporativa intermediária da mesma entidade citada acima; trocou o ticker de STLB3 para AORE3 em 2011 e posteriormente para SWET3 em 2015)",
    "": "",
    "": ""
}


excluidos_manual = base_pos_dfp[
    base_pos_dfp["TICKER"].isin(EXCLUSAO_MANUAL.keys())
].copy()
excluidos_manual["motivo_categoria"] = "EXCLUSAO_MANUAL"
excluidos_manual["motivo_detalhado"] = excluidos_manual["TICKER"].map(EXCLUSAO_MANUAL)
excluidos_manual["fonte_evidencia"] = "Revisão manual — 05_rebuilding_investable_universe_of_stocks.ipynb"

print(f"Excluídos manualmente: {len(excluidos_manual)}")
display(excluidos_manual[["TICKER", "codigo_cvm", "motivo_detalhado"]])

base_final = base_pos_dfp[
    ~base_pos_dfp["TICKER"].isin(EXCLUSAO_MANUAL.keys())
].copy()
print(f"\nUniverso final de tickers: {len(base_final)}")

Excluídos manualmente: 1


,TICKER,codigo_cvm,motivo_detalhado
557,PPLA11,80152,A PPLA Participations Ltd. é uma empresa estrangeira. Nã...



Universo final de tickers: 761


## Passo 5 — Consolidar e exportar a tabela de exclusões

Junta as três camadas de exclusão num único arquivo de auditoria, com rastreabilidade completa: qual
ticker saiu, em que camada, e com qual evidência.

In [14]:
pasta = pasta_interim / "08_investable_universe_of_stocks"
pasta.mkdir(parents=True, exist_ok=True)

colunas_exclusao = ["TICKER", "codigo_cvm", "motivo_categoria", "motivo_detalhado", "fonte_evidencia"]

excluidos_totais = pd.concat(
    [excluidos_sem_dfp[colunas_exclusao],
     excluidos_manual[colunas_exclusao]],
    ignore_index=True
).sort_values("TICKER")

excluidos_totais.to_csv(
    pasta / "excluded_tickers.csv", sep=";", index=False, encoding="utf-8-sig"
)

print(f"Total de tickers excluídos: {len(excluidos_totais)}")
print(f"Total de tickers no universo final: {len(base_final)}")
print(f"\nArquivo salvo em: {pasta / 'excluded_tickers.csv'}")

Total de tickers excluídos: 13
Total de tickers no universo final: 761

Arquivo salvo em: c:\Users\paulo\Desktop\Python\brazilian_financial_database\data\interim\08_investable_universe_of_stocks\excluded_tickers.csv


## Passo 6 — Reconstruir o mapeamento final e o JSON por trimestre

O mapeamento final (`ticker → codigo_cvm`) usa apenas os tickers aprovados e permanece em granularidade
ANUAL — a identidade da empresa não muda de um trimestre para outro. Já o universo investível por período é
reconstruído a partir do arquivo intermediário `ticker_per_period_raw.csv` (gerado em `03`, com granularidade
ticker×trimestre), filtrado pelos tickers do universo final. Isso preserva a informação de *em quais trimestres
exatos* cada ticker atendeu ao critério de liquidez mínima — não apenas o intervalo primeiro/último ano — o
que é essencial para casar corretamente com os fundamentos trimestrais (ITR) nas etapas seguintes do projeto.

In [15]:
# --- Mapeamento final (permanece anual) ---
colunas_mapping_final = [
    "TICKER", "codigo_cvm", "codigo_isin", "NOMRES",
    "primeiro_ano_observado", "ultimo_ano_observado"
]
mapping_final = base_final[colunas_mapping_final].sort_values("TICKER")
mapping_final.to_csv(
    pasta / "final_ticker_mapping.csv", sep=";", decimal=",", index=False, encoding="utf-8-sig"
)

# --- JSON por PERÍODO (trimestral) ---
ticker_periodo_raw = pd.read_csv(
    pasta_interim / "05_investable_universe_of_stocks/ticker_per_period_raw.csv",
    sep=";", dtype=str
)
ticker_periodo_raw["TICKER"] = ticker_periodo_raw["TICKER"].str.strip().str.upper()

ticker_periodo_final = ticker_periodo_raw[ticker_periodo_raw["TICKER"].isin(mapping_final["TICKER"])]

tickers_por_periodo_final = {
    periodo: grupo["TICKER"].unique().tolist()
    for periodo, grupo in ticker_periodo_final.groupby("periodo")
}
with open(pasta / "final_investable_stocks_per_period.json", "w") as f:
    json.dump(tickers_por_periodo_final, f, indent=4)

print(f"Mapeamento final salvo: {len(mapping_final)} tickers.")
print(f"JSON por período (trimestral) salvo: {pasta / 'final_investable_stocks_per_period.json'}")

Mapeamento final salvo: 761 tickers.
JSON por período (trimestral) salvo: c:\Users\paulo\Desktop\Python\brazilian_financial_database\data\interim\08_investable_universe_of_stocks\final_investable_stocks_per_period.json
